# Synthetic Data Evaluation

Evaluates synthetic stroke cohorts using the 12-metric framework (Yan 2022).
Compares BN, CTGAN, and hybrid approaches.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style='whitegrid', font_scale=1.2)

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

## 1. Generate Synthetic Data

In [ ]:
from src.models.bayesian_net import StrokeProfileBN
from src.models.ctgan_baseline import StrokeCTGAN

static = pd.read_parquet('../outputs/cohort/static_features.parquet')

# Fit BN
bn = StrokeProfileBN()
bn.fit(static)
bn_synth = bn.sample(n=len(static), seed=42)

# Fit CTGAN (on same features as BN)
feature_cols = bn._feature_cols
ctgan = StrokeCTGAN(epochs=50)
ctgan.fit(static[feature_cols])
ctgan_synth = ctgan.sample(n=len(static))

print(f'Real: {len(static)}, BN: {len(bn_synth)}, CTGAN: {len(ctgan_synth)}')

## 2. Fidelity — Marginal Distributions (KS Test)

In [ ]:
from src.evaluation.fidelity import dimension_wise_distribution

bn_ks = dimension_wise_distribution(static[feature_cols], bn_synth[feature_cols])
ctgan_ks = dimension_wise_distribution(static[feature_cols], ctgan_synth[feature_cols])

print('BN avg p-value:', f"{bn_ks['avg_pvalue']:.4f}")
print('CTGAN avg p-value:', f"{ctgan_ks['avg_pvalue']:.4f}")

# Plot per-column KS statistics
bn_stats = {k: v['ks_stat'] for k, v in bn_ks['per_column'].items()}
ctgan_stats = {k: v['ks_stat'] for k, v in ctgan_ks['per_column'].items()}

fig, ax = plt.subplots(figsize=(12, 5))
common_cols = sorted(set(bn_stats.keys()) & set(ctgan_stats.keys()))
x = np.arange(len(common_cols))
ax.bar(x - 0.2, [bn_stats[c] for c in common_cols], 0.4, label='BN', alpha=0.7)
ax.bar(x + 0.2, [ctgan_stats[c] for c in common_cols], 0.4, label='CTGAN', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(common_cols, rotation=45, ha='right')
ax.set_ylabel('KS Statistic (lower = better)')
ax.set_title('Marginal Distribution Fidelity')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/fidelity_ks.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Fidelity — Correlation Preservation

In [ ]:
from src.evaluation.fidelity import correlation_preservation

bn_corr = correlation_preservation(static[feature_cols], bn_synth[feature_cols])
ctgan_corr = correlation_preservation(static[feature_cols], ctgan_synth[feature_cols])

print(f"BN Frobenius distance: {bn_corr['frobenius_distance']:.4f}")
print(f"CTGAN Frobenius distance: {ctgan_corr['frobenius_distance']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(bn_corr['real_corr'], ax=axes[0], cmap='RdBu_r', center=0, vmin=-1, vmax=1)
axes[0].set_title('Real Data')
sns.heatmap(bn_corr['synth_corr'], ax=axes[1], cmap='RdBu_r', center=0, vmin=-1, vmax=1)
axes[1].set_title('BN Synthetic')
sns.heatmap(ctgan_corr['synth_corr'], ax=axes[2], cmap='RdBu_r', center=0, vmin=-1, vmax=1)
axes[2].set_title('CTGAN Synthetic')
plt.suptitle('Correlation Matrix Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/fidelity_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clinical Plausibility

In [ ]:
from src.evaluation.clinical_rules import check_clinical_rules

bn_rules = check_clinical_rules(bn_synth)
ctgan_rules = check_clinical_rules(ctgan_synth)
real_rules = check_clinical_rules(static)

print('Clinical Rule Violations:')
print(f"  Real:  {real_rules['total_violations']} ({real_rules['total_violation_rate']*100:.2f}%)")
print(f"  BN:    {bn_rules['total_violations']} ({bn_rules['total_violation_rate']*100:.2f}%)")
print(f"  CTGAN: {ctgan_rules['total_violations']} ({ctgan_rules['total_violation_rate']*100:.2f}%)")

## 5. Discriminator Score

In [ ]:
from src.evaluation.fidelity import discriminator_score

bn_disc = discriminator_score(static[feature_cols], bn_synth[feature_cols])
ctgan_disc = discriminator_score(static[feature_cols], ctgan_synth[feature_cols])

print(f"BN discriminator AUC: {bn_disc['auc']:.4f} (ideal: 0.5)")
print(f"CTGAN discriminator AUC: {ctgan_disc['auc']:.4f} (ideal: 0.5)")

## 6. Rare Event Preservation (Stroke Subtypes)

In [ ]:
from src.evaluation.fidelity import medical_concept_abundance

if 'stroke_subtype' in bn_synth.columns:
    bn_mca = medical_concept_abundance(static, bn_synth, 'stroke_subtype')
    print(f"BN stroke subtype Manhattan distance: {bn_mca['manhattan_distance']:.4f}")
    print(f"  Real: {bn_mca['real_dist']}")
    print(f"  BN:   {bn_mca['synth_dist']}")

## 7. Privacy Metrics

In [ ]:
from src.evaluation.privacy import membership_inference_attack, nearest_neighbor_distance

bn_mia = membership_inference_attack(static[feature_cols], bn_synth[feature_cols])
bn_dcr = nearest_neighbor_distance(static[feature_cols], bn_synth[feature_cols])

print(f"BN MIA F1: {bn_mia['mia_f1']:.4f} (threshold: < 0.20)")
print(f"BN mean DCR: {bn_dcr['mean_dcr']:.4f}")